# Percentile thresholding & aggregated SDM outputs (prototype)

Notebook-only exploration: convert continuous **0–1** suitability rasters into binary layers and **potential** species-count surfaces (roost, in-flight, combined without double-counting per species). **Runs do not write CSV, GeoTIFF, or PNG** — results stay as DataFrames/plots/objects in-memory.

**Not** observed richness or confirmed presence.

Helper functions are defined here so they can move into modules later.


## 1. Setup and configuration

Paths and threshold labels are in the next code cell. **Nothing is written to disk**—tables and rasters stay in memory for exploration. Raster bands use `band_key` = `get_model_id([latin_name, activity_type])` (see §2).

`PLOT_STEP` subsamples the prediction raster for maps and binary/aggregate math (faster, lower memory); increase it for large grids.


In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
from IPython.display import display
from rasterio.enums import Resampling
from rasterio.windows import Window

from sdm.utils.io import load_pickled_model, set_project_wd
from sdm.commands.modelling.utils import get_model_id
from sdm.models.core.pipeline_features import pipeline_selected_feature_names

set_project_wd(verbose=False)

REPO_ROOT = Path.cwd()
PATH_MODEL_RESULTS = REPO_ROOT / "data" / "sdm_models" / "model_results.csv"
PATH_TRAINING = REPO_ROOT / "data" / "sdm_models" / "training_data.parquet"
PATH_PREDICTIONS = REPO_ROOT / "data" / "sdm_predictions" / "all_predictions.tif"

# Subsample factor for in-notebook rasters (maps / binary / aggregate); no files written.
PLOT_STEP = 16

THRESHOLD_COLS = {
    "p00_min_presence": "threshold_p00_min_presence",
    "p10_main": "threshold_p10_main",
    "p25_conservative": "threshold_p25_conservative",
}

plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 120})


## 2. Load and inspect source artefacts


In [ ]:
def load_model_results(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing model index: {path}")
    df = pd.read_csv(path)
    df["band_key"] = df.apply(
        lambda r: get_model_id([r["latin_name"], r["activity_type"]]), axis=1
    )
    return df


def load_training_data(path: Path) -> pd.DataFrame | gpd.GeoDataFrame:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing training parquet: {path}. Run `sdm train` (writes training_data.parquet)."
        )
    try:
        return gpd.read_parquet(path)
    except Exception:
        return pd.read_parquet(path)


def inspect_prediction_raster(path: Path) -> dict:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prediction raster: {path}. Run `sdm predict` (writes all_predictions.tif)."
        )
    with rio.open(path) as src:
        return {
            "path": path,
            "count": src.count,
            "crs": src.crs,
            "transform": src.transform,
            "bounds": src.bounds,
            "shape": (src.height, src.width),
            "nodata": src.nodata,
            "descriptions": list(src.descriptions) if src.descriptions else [],
            "dtypes": [src.dtypes[i] for i in range(1, src.count + 1)],
        }


model_results = load_model_results(PATH_MODEL_RESULTS)
training_df = load_training_data(PATH_TRAINING)
pred_info = inspect_prediction_raster(PATH_PREDICTIONS)

display(model_results.head())
print("Training columns:", list(training_df.columns))
print("Training rows:", len(training_df))
if "class" in training_df.columns:
    print(
        "Presence:",
        int((training_df["class"] == 1).sum()),
        "Background:",
        int((training_df["class"] == 0).sum()),
    )
if "identifier" in training_df.columns:
    print("Unique training identifiers:", training_df["identifier"].nunique())
print("Raster shape:", pred_info["shape"], "bands:", pred_info["count"], "nodata:", pred_info["nodata"])
print("CRS:", pred_info["crs"])
print("Band descriptions:", pred_info["descriptions"])


### Identifier alignment

- `identifier` in CSV and training (e.g. `Nyctalus noctula_In flight`)
- `band_key` = `get_model_id([latin_name, activity_type])` matches **raster band descriptions** from `sdm predict` (`nyctalus_noctula_in_flight`)


In [ ]:
model_ids_from_results = set(model_results["identifier"])
model_ids_from_training = (
    set(training_df["identifier"].unique()) if "identifier" in training_df.columns else set()
)
model_ids_from_raster_bands = {d for d in pred_info["descriptions"] if d}
band_keys_from_results = set(model_results["band_key"])

in_results_and_training = model_ids_from_results & model_ids_from_training
band_keys_matched = sorted(band_keys_from_results & model_ids_from_raster_bands)

mr_use = (
    model_results[
        model_results["identifier"].isin(in_results_and_training)
        & model_results["band_key"].isin(band_keys_matched)
    ]
    .copy()
    .sort_values(["activity_type", "latin_name"])
    .reset_index(drop=True)
)

print(
    "Identifiers in both CSV and training:",
    len(in_results_and_training),
    "| band_key also in raster:",
    len(band_keys_matched),
)
print("CSV ids missing from training:", model_ids_from_results - model_ids_from_training)
print(
    "CSV band_key missing from raster:",
    sorted(band_keys_from_results - model_ids_from_raster_bands),
)
print("Training ids missing from CSV:", model_ids_from_training - model_ids_from_results)

if not band_keys_matched:
    raise RuntimeError("No raster bands match model_results; stopping.")

if (band_keys_from_results - model_ids_from_raster_bands) or (
    model_ids_from_results - model_ids_from_training
):
    print("\n*** WARNING: using intersection only; fix data or regenerate predictions. ***\n")

raster_band_keys_ordered = [
    d for d in pred_info["descriptions"] if d in band_keys_matched
]
print("Matched models (raster band order):", len(raster_band_keys_ordered))
mr_by_band = mr_use.set_index("band_key")

for d in raster_band_keys_ordered:
    if d not in mr_by_band.index:
        raise KeyError(f"Raster band description {d!r} has no row in model_results / training intersection")

band_axes = (
    mr_use.drop_duplicates(subset=["band_key"], keep="first")
    .set_index("band_key")
    .loc[raster_band_keys_ordered]
    [["identifier", "latin_name", "activity_type"]]
    .assign(band_ix=np.arange(len(raster_band_keys_ordered)))
)
display(band_axes)



### Notebook utilities (`xarray`, vectorised masking)

Everything below stays local to this notebook; short functions keep section cells declarative.

In [ ]:
def continuous_valid_mask(cube_nb_hw: np.ndarray, nodata: float | None) -> np.ndarray:
    ok = np.isfinite(cube_nb_hw)
    if nodata is not None and not (isinstance(nodata, float) and np.isnan(nodata)):
        ok &= cube_nb_hw != nodata
    return ok


def predictions_subsampled_array(
    raster_path: Path,
    band_keys_ordered: list[str],
    descriptions: list[str],
    plot_step: int,
) -> xr.DataArray:
    idxs = [descriptions.index(bk) + 1 for bk in band_keys_ordered]
    with rio.open(raster_path) as src:
        nodata = src.nodata
        hs, ws = max(1, src.height // plot_step), max(1, src.width // plot_step)
        stacked = np.stack(
            [
                src.read(ix, out_shape=(hs, ws), resampling=Resampling.average).astype(
                    np.float64
                )
                for ix in idxs
            ],
            axis=0,
        )
    return xr.DataArray(
        stacked,
        dims=("band_key", "y", "x"),
        coords={"band_key": band_keys_ordered},
        name="prediction",
        attrs={"nodata": nodata},
    )


def binary_stacks_vectorised(
    cont: xr.DataArray,
    band_keys_ordered: list[str],
    thresh_table: pd.DataFrame,
    threshold_cols_map: dict[str, str],
    bin_nodata: int,
) -> dict[str, xr.DataArray]:
    th_aligned = thresh_table.set_index("band_key").reindex(band_keys_ordered)
    cube = cont.values.astype(np.float64)
    nod = cont.attrs["nodata"]
    valid_hw = continuous_valid_mask(cube, nod)
    out: dict[str, xr.DataArray] = {}
    for label, col in threshold_cols_map.items():
        tvec = pd.to_numeric(th_aligned[col], errors="coerce").to_numpy(dtype=np.float64).reshape(
            -1, 1, 1
        )
        miss_b = np.isnan(tvec)
        hi = np.full(cube.shape, bin_nodata, dtype=np.uint8)
        ok = (~miss_b) & valid_hw
        t_b = np.broadcast_to(tvec, cube.shape)
        hi[ok] = (cube[ok] >= t_b[ok]).astype(np.uint8)
        out[label] = xr.DataArray(hi, coords=cont.coords, dims=cont.dims, attrs={"scenario": label})
    return out


def aggregate_species_layers(
    binary_cube_nb_hw: np.ndarray,
    band_axes_tbl: pd.DataFrame,
    bin_nodata: int,
    agg_fill: int = 65535,
) -> dict[str, np.ndarray]:
    vf = np.uint16(agg_fill)
    valid_px = (binary_cube_nb_hw != bin_nodata).any(axis=0)
    ro = band_axes_tbl.activity_type.eq("Roost").to_numpy()
    fl = band_axes_tbl.activity_type.eq("In flight").to_numpy()

    rs = (
        binary_cube_nb_hw[ro].sum(axis=0).astype(np.uint16)
        if ro.any()
        else np.zeros(valid_px.shape, np.uint16)
    )
    fc = (
        binary_cube_nb_hw[fl].sum(axis=0).astype(np.uint16)
        if fl.any()
        else np.zeros(valid_px.shape, np.uint16)
    )

    tot = np.zeros(valid_px.shape, dtype=np.uint16)
    for _, g in band_axes_tbl.groupby("latin_name", sort=False):
        tot += binary_cube_nb_hw[g["band_ix"].to_numpy()].max(axis=0).astype(np.uint16)

    def mask_to_fill(x):
        return np.where(valid_px, x, vf)

    return {"all": mask_to_fill(tot), "roost": mask_to_fill(rs), "in_flight": mask_to_fill(fc)}


def add_subsampled_pixel_qa(
    qa: pd.DataFrame,
    binaries: dict[str, np.ndarray],
    band_keys_ordered: list[str],
    bin_nodata: int,
) -> pd.DataFrame:
    key_to_ix = {bk: i for i, bk in enumerate(band_keys_ordered)}
    for tlab, cube in binaries.items():
        sel = qa["threshold_label"].eq(tlab)
        for ix in qa.index[sel]:
            bk = qa.at[ix, "band_key"]
            band = cube[key_to_ix[bk]].ravel()
            valid = band != bin_nodata
            suited = valid & (band == 1)
            qa.at[ix, "suitable_cell_count"] = int(suited.sum())
            qa.at[ix, "valid_cell_count"] = int(valid.sum())
            v = valid.sum()
            qa.at[ix, "suitable_area_percent"] = (
                100.0 * suited.sum() / float(v) if v else np.nan
            )
    return qa


def merge_secondary_qa_warnings(qa: pd.DataFrame) -> pd.DataFrame:
    extra = []
    for _, row in qa.iterrows():
        p = []
        if row["threshold_label"] == "p10_main":
            eo = row["training_omission_rate"]
            if pd.notna(eo) and abs(float(eo) - 0.10) > 0.08:
                p.append("omission_not_near_10pct")
            sp = row["suitable_area_percent"]
            if pd.notna(sp):
                if float(sp) > 80:
                    p.append("suitable_pct>80")
                if float(sp) < 1:
                    p.append("suitable_pct<1")
        extra.append("; ".join(p))
    add = ["; " + e if e else "" for e in extra]
    out = qa.copy()
    out["warning"] = out["warning"].fillna("") + add
    return out

def modeling_quality_warnings(n_presence: int, mean_cv) -> list[str]:
    w = []
    if n_presence < 30:
        w.append("n_presence<30")
    if pd.isna(mean_cv):
        w.append("mean_cv_missing")
    elif float(mean_cv) < 0.7:
        w.append("mean_cv<0.7")
    return w


def presence_suitability_scores(model_path: str | Path, sub_pr) -> np.ndarray | None:
    model = load_pickled_model(model_path)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            X = sub_pr.drop(columns=["geometry"], errors="ignore").copy()
            return model.predict_proba(X)[:, 1]
        except Exception:
            fn = pipeline_selected_feature_names(model)
            return model.predict_proba(sub_pr.loc[:, fn])[:, 1]


def summarise_scores(scores: np.ndarray) -> dict[str, float]:
    return {
        "presence_score_min": float(scores.min()),
        "presence_score_p10": float(np.percentile(scores, 10)),
        "presence_score_p25": float(np.percentile(scores, 25)),
        "presence_score_median": float(np.percentile(scores, 50)),
        "presence_score_mean": float(scores.mean()),
        "presence_score_max": float(scores.max()),
        "threshold_p00_min_presence": float(scores.min()),
        "threshold_p10_main": float(np.percentile(scores, 10)),
        "threshold_p25_conservative": float(np.percentile(scores, 25)),
    }


## 3. Presence-based percentile thresholds


In [ ]:
# Presence scores → threshold table (keeps ``score_arrays`` for omission QA)
score_arrays: dict[str, np.ndarray] = {}

rows_out = []
for _, row in mr_use.iterrows():
    ident = row["identifier"]
    sub_all = training_df.loc[training_df["identifier"].eq(ident)]
    sub_pr = sub_all.loc[sub_all["class"].eq(1)]
    warn_parts = modeling_quality_warnings(len(sub_pr), row.get("mean_cv_score"))

    scores = None
    try:
        scores = presence_suitability_scores(row["model_path"], sub_pr)
    except Exception as e:
        warn_parts.append(f"predict_fail:{e!r}")
    if scores is None or scores.size == 0:
        warn_parts.append("no_scores")
        stats = {k: np.nan for k in (
            "presence_score_min", "presence_score_p10", "presence_score_p25",
            "presence_score_median", "presence_score_mean", "presence_score_max",
            "threshold_p00_min_presence", "threshold_p10_main", "threshold_p25_conservative",
        )}
    else:
        score_arrays[ident] = scores
        stats = summarise_scores(scores)

    rows_out.append({
        **stats,
        "identifier": ident,
        "band_key": row["band_key"],
        "latin_name": row["latin_name"],
        "activity_type": row["activity_type"],
        "n_presence": len(sub_pr),
        "n_background": int(sub_all["class"].eq(0).sum()),
        "mean_cv_score": row.get("mean_cv_score"),
        "std_cv_score": row.get("std_cv_score"),
        "warning": "; ".join(warn_parts),
    })

threshold_summary = pd.DataFrame(rows_out).sort_values(["activity_type", "latin_name"])
display(threshold_summary)


## 4. Presence score distributions


In [ ]:
thresh_by_bk = threshold_summary.set_index("band_key")
nplots = len(raster_band_keys_ordered)
ncol = min(4, max(3, int(np.ceil(np.sqrt(nplots)))))
nrow = int(np.ceil(nplots / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3.4, nrow * 2.6), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for i, bk in enumerate(raster_band_keys_ordered):
    ax = axes.flat[i]
    row = thresh_by_bk.loc[bk]
    ident = row["identifier"]
    sc = score_arrays.get(ident)
    ax.set_axis_on()
    if sc is None or sc.size == 0:
        ax.text(0.5, 0.5, "no scores", ha="center")
        continue
    ax.hist(sc, bins=min(40, max(10, len(sc) // 2)), density=True, color="#4C72B0", alpha=0.75)
    for x, ls in [(row["threshold_p00_min_presence"], ":"), (row["threshold_p10_main"], "-"), (row["threshold_p25_conservative"], "--")]:
        if pd.notna(x):
            ax.axvline(float(x), color="k", ls=ls, lw=1.5)
    mcv = row["mean_cv_score"]
    ttl = row["latin_name"] + chr(10) + f"{row['activity_type']} · n={int(row['n_presence'])}"
    if pd.notna(mcv):
        ttl += f" · CV={float(mcv):.2f}"
    ax.set_title(ttl, fontsize=8)
fig.suptitle("Presence-site predicted suitability with p00 (:), p10 (—), p25 (--)", fontsize=10)
plt.tight_layout()
plt.show()

# Thresholds by species (p10) — one row per model
fig2, ax2 = plt.subplots(figsize=(10, max(4, 0.22 * len(threshold_summary))))
y = np.arange(len(threshold_summary))
ax2.barh(y, threshold_summary["threshold_p10_main"].fillna(0), color="#55A868")
ax2.set_yticks(y)
ax2.set_yticklabels(
    threshold_summary["latin_name"] + " — " + threshold_summary["activity_type"],
    fontsize=7,
)
ax2.set_xlabel("p10 threshold (presence scores)")
ax2.set_title("Per-model p10 threshold")
plt.tight_layout()
plt.show()


## 5. Threshold continuous predictions (in memory, subsampled)

Read matched bands from `all_predictions.tif` at reduced resolution (`PLOT_STEP` via `Resampling.average`), then build binary stacks per threshold scenario. **Nothing is saved to GeoTIFF.**


In [ ]:
th_lookup = threshold_summary.set_index("band_key")
BIN_NODATA = 255

pred_continuous = predictions_subsampled_array(
    PATH_PREDICTIONS, raster_band_keys_ordered, pred_info["descriptions"], PLOT_STEP
)
EXPLORE_GRID_SHAPE = (int(pred_continuous.sizes["y"]), int(pred_continuous.sizes["x"]))

_binary_xr = binary_stacks_vectorised(
    pred_continuous, raster_band_keys_ordered, threshold_summary, THRESHOLD_COLS, BIN_NODATA
)
binary_explore = {k: da.values for k, da in _binary_xr.items()}
pred_continuous.shape, {k: v.shape for k, v in binary_explore.items()}


## 6. Aggregate counts (in memory)

Roost-only, in-flight-only, and combined (max per species, then sum) layers are stored in **`aggregate_explore`** — **no raster files written.**



In [ ]:
aggregate_explore = {
    lab: aggregate_species_layers(binary_explore[lab], band_axes, BIN_NODATA)
    for lab in THRESHOLD_COLS
}
sorted(aggregate_explore.keys())


**Combined count rule:** for each species, `suitable = max(binary_roost, binary_in_flight)` (missing activity treated as 0), then sum over species. Roost / in-flight maps sum binary bands for that activity only (one model per species–activity).


## 7. QA summary (omission & suitable area on subsampled grid)

Training omission uses full presence-site scores. `suitable_cell_count` / `suitable_area_percent` reflect the **subsampled** binary stacks (`PLOT_STEP`), not full-resolution rasters — useful for comparative QA only.


In [ ]:
_id_cols = ["identifier", "band_key", "latin_name", "activity_type",
              "n_presence", "mean_cv_score", "warning"]
_thresh_cols = list(THRESHOLD_COLS.values())
long_df = threshold_summary.melt(
    id_vars=_id_cols,
    value_vars=_thresh_cols,
    var_name="_tc",
    value_name="threshold",
)
_lab_map = {v: k for k, v in THRESHOLD_COLS.items()}
long_df["threshold_label"] = long_df["_tc"].map(_lab_map)
long_df.drop(columns="_tc", inplace=True)


long_df["training_omission_rate"] = long_df.apply(
    lambda r: (
        float((score_arrays[r["identifier"]] < float(r["threshold"])).mean())
        if pd.notna(r["threshold"])
        and score_arrays.get(r["identifier"]) is not None
        and score_arrays[r["identifier"]].size
        else np.nan
    ),
    axis=1,
)


qa_df = add_subsampled_pixel_qa(long_df, binary_explore, raster_band_keys_ordered, BIN_NODATA)
qa_df = merge_secondary_qa_warnings(qa_df)
display(qa_df.head(20))


## 8. Map QA figures (from in-memory aggregates)

Figures are shown only in the notebook (no PNG export).


In [ ]:
AGG_NODATA = 65535


def plot_count_array(arr: np.ndarray, title: str) -> None:
    a = arr.astype(np.float64)
    a = np.ma.masked_where(a >= AGG_NODATA - 1, a)
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(a, cmap="viridis")
    plt.colorbar(im, ax=ax, shrink=0.7, label="Potential modelled species count")
    ax.set_title(title + chr(10) + "(note: suitability-based index, not observed richness)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_count_array(
    aggregate_explore["p10_main"]["all"],
    "Potential species count (all activities combined, p10, subsampled)",
)
plot_count_array(
    aggregate_explore["p10_main"]["roost"],
    "Roost-only count (p10, subsampled)",
)
plot_count_array(
    aggregate_explore["p10_main"]["in_flight"],
    "In-flight-only count (p10, subsampled)",
)

# Sensitivity triptych
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, lab, ttl in zip(
    axes,
    ["p00_min_presence", "p10_main", "p25_conservative"],
    ["p00 min presence", "p10 main", "p25 conservative"],
):
    arr = aggregate_explore[lab]["all"].astype(np.float64)
    arr = np.ma.masked_where(arr >= AGG_NODATA - 1, arr)
    im = ax.imshow(arr, cmap="viridis")
    ax.set_title(ttl)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Sensitivity: potential species count (all) across thresholds (subsampled)")
plt.tight_layout()
plt.show()


## 9. Optional: contributions at one high-count pixel


In [ ]:
# Pick a high-count pixel on the subsampled p10 "all" layer, then sample full-resolution EV stack at mapped location

z = aggregate_explore["p10_main"]["all"]
vm = (z < AGG_NODATA - 1) & (z > 0)
if vm.any():
    flat = np.argmax(z * vm)
    r, c = np.unravel_index(flat, z.shape)
    H, W = pred_info["shape"]
    hs, ws = EXPLORE_GRID_SHAPE
    r_full = min(int((r + 0.5) * H / hs), H - 1)
    c_full = min(int((c + 0.5) * W / ws), W - 1)
    win = Window(c_full, r_full, 1, 1)
    print(f"Subsampling grid (row, col)=({r}, {c}); approx full-res centred near (row,col)=({r_full}, {c_full})")

    with rio.open(PATH_PREDICTIONS) as srcp:
        contig = np.array(
            [srcp.read(i + 1, window=win)[0, 0] for i in range(len(pred_info["descriptions"]))]
        )
    rows_demo = []
    for bk in raster_band_keys_ordered:
        ib = pred_info["descriptions"].index(bk)
        score = float(contig[ib])
        tr = th_lookup.loc[bk]
        thr = float(tr["threshold_p10_main"])
        rows_demo.append(
            {
                "Location": f"rc_full_{r_full}_{c_full}",
                "Species": tr["latin_name"],
                "Activity": tr["activity_type"],
                "Continuous score": score,
                "Threshold (p10)": thr,
                "Suitable?": score >= thr,
            }
        )
    display(pd.DataFrame(rows_demo))
else:
    print("No suitable pixels on subsampled aggregate — skip drill-down.")


## 10. Conclusions (edit after run)

- **Alignment:** confirm whether `band_key` matched all intended models; review any warnings from §2.
- **Threshold choice:** inspect `threshold_summary` / `qa_df` in-session; omission rates vs `suitable_area_percent` (on subsampled grids); p10 is a convention, not biology.
- **Problematic models:** rows with `warning` / low `n_presence` / low CV.
- **Sensitivity:** compare the three panels in §8 (`aggregate_explore` across thresholds).
- **Next steps for production:** if you need reproducible artefacts, persist tables/rasters deliberately (this notebook avoids disk writes).

**Caveats**

- Maps and binary stats here use **`PLOT_STEP` subsampling** — not identical to native-resolution production outputs.
- Maps show *potential* modelled suitability, not confirmed bats.
- Aggregate counts are model-derived; do not report as field-observed richness.

### Code organisation notes

- **`band_axes`**: single table for raster-band order (`latin_name`, `activity_type`, `band_ix`) — avoids rebuilding dict/index maps in every section.
- **`xarray`**: named dimensions `("band_key", "y", "x")` for subsampled grids; slicing by activity/roost reduces index bookkeeping.
- Helpers keep loops minimal: vectorised masking for binary layers, pandas `explode`/merge paths for QA, small pure functions grouped before §3.
